# Using DeterministicModel: Minimal Working Example

This tutorial demonstrates how to use `DeterministicModel` for optimization problems where some objectives or constraints are analytically known. This addresses the use case mentioned in GitHub issues #935 and #1192 where users want to optimize mixed analytical and black-box functions.

## Problem Overview

Common scenario in physical sciences and engineering:
- **Analytical functions**: Cost, energy, or other computable objectives/constraints
- **Black-box functions**: Complex simulations or experimental results
- **Goal**: Efficiently optimize without learning surrogate models for known functions

## Key Concepts

1. **`GenericDeterministicModel`**: Wraps analytical functions for BoTorch
2. **`ModelList`**: Combines deterministic and probabilistic models
3. **Integration**: How to use these in optimization workflows

## Part 1: Basic DeterministicModel Usage

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from typing import Callable

# Core BoTorch imports
from botorch.models.deterministic import GenericDeterministicModel
from botorch.models import SingleTaskGP
from botorch.models.model import ModelList
from botorch.acquisition import ExpectedImprovement
from botorch.optim import optimize_acqf
from botorch.utils.datasets import SupervisedDataset

print("Imports successful!")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## Step 1: Define Analytical and Black-box Functions

In [ ]:
def analytical_cost_function(x: torch.Tensor) -> torch.Tensor:
    """
    Analytical cost function: f(x,y) = x² + y²
    This could represent monetary cost, energy consumption, etc.
    
    Args:
        x: Input tensor of shape (..., 2) where columns are [x, y]
    
    Returns:
        Cost values of shape (..., 1)
    """
    return (x**2).sum(dim=-1, keepdim=True)

def black_box_performance(x: torch.Tensor) -> torch.Tensor:
    """
    Black-box performance function that we want to maximize.
    This simulates an expensive-to-evaluate function like simulation results.
    
    For demonstration: f(x,y) = -(x-0.3)² - (y-0.3)² + 0.5 + noise
    """
    base_value = -((x[..., 0:1] - 0.3)**2 + (x[..., 1:2] - 0.3)**2) + 0.5
    noise = 0.1 * torch.randn_like(base_value)
    return base_value + noise

def constraint_function(x: torch.Tensor) -> torch.Tensor:
    """
    Black-box constraint function: g(x,y) = x + y - 0.8 ≤ 0
    """
    return x.sum(dim=-1, keepdim=True) - 0.8 + 0.05 * torch.randn(x.shape[0], 1)

# Test the functions
test_points = torch.tensor([[0.2, 0.3], [0.5, 0.4], [0.1, 0.1]])
print(f"Test points: {test_points}")
print(f"Analytical cost: {analytical_cost_function(test_points).flatten()}")
print(f"Black-box performance: {black_box_performance(test_points).flatten()}")
print(f"Constraint values: {constraint_function(test_points).flatten()}")

## Step 2: Create DeterministicModel

In [ ]:
# Create a DeterministicModel for the analytical cost function
deterministic_cost_model = GenericDeterministicModel(f=analytical_cost_function)

print("DeterministicModel created!")
print(f"Model class: {type(deterministic_cost_model).__name__}")

# Test the deterministic model
test_input = torch.tensor([[0.4, 0.5]])
analytical_result = analytical_cost_function(test_input)
model_result = deterministic_cost_model(test_input)

print(f"\nDirect function call: {analytical_result.item():.6f}")
print(f"DeterministicModel call: {model_result.item():.6f}")
print(f"Results match: {torch.allclose(analytical_result, model_result)}")

# Key insight: DeterministicModel returns a Tensor directly
print(f"\nResult type: {type(model_result).__name__}")
print(f"Result shape: {model_result.shape}")
print(f"No uncertainty (deterministic): {model_result}")

## Step 3: Create Mixed Model with ModelList

In [ ]:
# Generate some initial data for the black-box functions
n_initial = 8
bounds = torch.tensor([[0.0, 0.0], [1.0, 1.0]], dtype=torch.double)

# Sobol sampling for initial points
from torch.quasirandom import SobolEngine
sobol = SobolEngine(dimension=2, scramble=True, seed=42)
initial_X = bounds[0] + (bounds[1] - bounds[0]) * sobol.draw(n_initial)

# Evaluate black-box functions at initial points
initial_performance = black_box_performance(initial_X)
initial_constraints = constraint_function(initial_X)

print(f"Generated {n_initial} initial points")
print(f"X shape: {initial_X.shape}")
print(f"Performance shape: {initial_performance.shape}")
print(f"Constraints shape: {initial_constraints.shape}")

# Create GP models for black-box functions
performance_gp = SingleTaskGP(initial_X, initial_performance)
constraint_gp = SingleTaskGP(initial_X, initial_constraints)

# Create ModelList combining deterministic and probabilistic models
mixed_model = ModelList(
    deterministic_cost_model,  # Model 0: Analytical cost (deterministic)
    performance_gp,            # Model 1: Performance (probabilistic)
    constraint_gp              # Model 2: Constraint (probabilistic)
)

print(f"\nMixed model created with {len(mixed_model.models)} sub-models:")
for i, model in enumerate(mixed_model.models):
    print(f"  Model {i}: {type(model).__name__}")

# Test the mixed model - access individual models
test_X = torch.tensor([[0.3, 0.4]])
cost_output = mixed_model.models[0](test_X)  # Deterministic model
performance_output = mixed_model.models[1].posterior(test_X)  # GP model
constraint_output = mixed_model.models[2].posterior(test_X)   # GP model

print(f"\nMixed model output for {test_X.flatten().tolist()}:")
print(f"  Cost (deterministic): {cost_output.item():.4f}")
print(f"  Performance (GP mean): {performance_output.mean.item():.4f}")
print(f"  Performance (GP std): {performance_output.variance.sqrt().item():.4f}")
print(f"  Constraint (GP mean): {constraint_output.mean.item():.4f}")
print(f"  Constraint (GP std): {constraint_output.variance.sqrt().item():.4f}")

## Step 4: Multi-objective Optimization Example

Now let's demonstrate how to use the mixed model for optimization where we want to:
- Minimize analytical cost (deterministic)
- Maximize black-box performance (probabilistic)
- Subject to constraint ≤ 0

In [ ]:
from botorch.acquisition.multi_objective import qExpectedHypervolumeImprovement
from botorch.utils.multi_objective.box_decompositions.non_dominated import NondominatedPartitioning
from botorch.utils.multi_objective.pareto import is_non_dominated
from botorch.utils.sampling import sample_simplex

def multi_objective_optimization_step(model: ModelList, 
                                    current_X: torch.Tensor,
                                    bounds: torch.Tensor,
                                    ref_point: torch.Tensor):
    """
    Perform one step of multi-objective optimization.
    """
    # Current objectives (cost, performance)
    current_cost = analytical_cost_function(current_X)  # Minimize
    current_performance = black_box_performance(current_X)  # Maximize
    
    # For multi-objective, we need to negate what we want to maximize
    current_objectives = torch.cat([
        current_cost,           # Minimize cost
        -current_performance    # Minimize negative performance (= maximize performance)
    ], dim=-1)
    
    # Find non-dominated points
    pareto_mask = is_non_dominated(current_objectives)
    pareto_Y = current_objectives[pareto_mask]
    
    print(f"Found {pareto_mask.sum()} Pareto-optimal points out of {len(current_X)}")
    
    # Create partitioning for hypervolume
    partitioning = NondominatedPartitioning(ref_point=ref_point, Y=pareto_Y)
    
    # Create acquisition function (using only cost and performance models)
    objective_model = ModelList(model.models[0], model.models[1])  # Cost + Performance
    
    acq_func = qExpectedHypervolumeImprovement(
        model=objective_model,
        ref_point=ref_point,
        partitioning=partitioning,
    )
    
    # Optimize acquisition function
    candidates, acq_value = optimize_acqf(
        acq_function=acq_func,
        bounds=bounds,
        q=1,
        num_restarts=10,
        raw_samples=100,
    )
    
    return candidates, acq_value.item()

# Set up multi-objective optimization
ref_point = torch.tensor([2.0, -0.8])  # Reference point for hypervolume

print("Starting multi-objective optimization...")
print(f"Reference point: {ref_point.tolist()}")
print("Objectives: [minimize cost, minimize -performance]")

# Perform one optimization step
try:
    next_point, acq_val = multi_objective_optimization_step(
        mixed_model, initial_X, bounds, ref_point
    )
    
    print(f"\nNext recommended point: {next_point.flatten().tolist()}")
    print(f"Acquisition value: {acq_val:.6f}")
    
    # Evaluate the recommended point
    next_cost = analytical_cost_function(next_point)
    next_performance = black_box_performance(next_point)  # This would be expensive in practice
    next_constraint = constraint_function(next_point)
    
    print(f"\nPredicted values at recommended point:")
    print(f"  Cost (analytical): {next_cost.item():.4f}")
    print(f"  Performance: {next_performance.item():.4f}")
    print(f"  Constraint: {next_constraint.item():.4f} (should be ≤ 0)")
    print(f"  Feasible: {next_constraint.item() <= 0}")
    
except Exception as e:
    print(f"Optimization step failed: {e}")
    print("This is often due to insufficient data or numerical issues.")

## Step 5: Visualization

In [ ]:
# Create a visualization of the problem
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Create a grid for plotting
x_range = np.linspace(0, 1, 50)
y_range = np.linspace(0, 1, 50)
X_grid, Y_grid = np.meshgrid(x_range, y_range)
grid_points = torch.tensor(np.column_stack([X_grid.ravel(), Y_grid.ravel()]), dtype=torch.double)

# Evaluate functions on the grid
cost_grid = analytical_cost_function(grid_points).numpy().reshape(X_grid.shape)
performance_grid = black_box_performance(grid_points).numpy().reshape(X_grid.shape)
constraint_grid = constraint_function(grid_points).numpy().reshape(X_grid.shape)

# Plot 1: Analytical cost function (deterministic)
im1 = axes[0, 0].contourf(X_grid, Y_grid, cost_grid, levels=20, cmap='viridis')
axes[0, 0].scatter(initial_X[:, 0], initial_X[:, 1], c='red', s=50, alpha=0.7, label='Initial points')
axes[0, 0].set_title('Analytical Cost Function\n(DeterministicModel)')
axes[0, 0].set_xlabel('x')
axes[0, 0].set_ylabel('y')
axes[0, 0].legend()
plt.colorbar(im1, ax=axes[0, 0])

# Plot 2: Black-box performance function (with GP uncertainty)
im2 = axes[0, 1].contourf(X_grid, Y_grid, performance_grid, levels=20, cmap='plasma')
axes[0, 1].scatter(initial_X[:, 0], initial_X[:, 1], c='red', s=50, alpha=0.7, label='Training data')
axes[0, 1].set_title('Black-box Performance Function\n(Gaussian Process)')
axes[0, 1].set_xlabel('x')
axes[0, 1].set_ylabel('y')
axes[0, 1].legend()
plt.colorbar(im2, ax=axes[0, 1])

# Plot 3: Constraint function
im3 = axes[1, 0].contourf(X_grid, Y_grid, constraint_grid, levels=[-2, 0, 2], colors=['green', 'red'], alpha=0.3)
axes[1, 0].contour(X_grid, Y_grid, constraint_grid, levels=[0], colors=['black'], linewidths=2)
axes[1, 0].scatter(initial_X[:, 0], initial_X[:, 1], c='red', s=50, alpha=0.7)
axes[1, 0].set_title('Constraint Function\n(Green=Feasible, Red=Infeasible)')
axes[1, 0].set_xlabel('x')
axes[1, 0].set_ylabel('y')

# Plot 4: Pareto front in objective space
current_cost = analytical_cost_function(initial_X).numpy().flatten()
current_performance = black_box_performance(initial_X).numpy().flatten()
current_constraints = constraint_function(initial_X).numpy().flatten()

# Color points by feasibility
feasible = current_constraints <= 0
axes[1, 1].scatter(current_cost[feasible], current_performance[feasible], 
                  c='green', s=50, alpha=0.7, label='Feasible')
axes[1, 1].scatter(current_cost[~feasible], current_performance[~feasible], 
                  c='red', s=50, alpha=0.7, label='Infeasible')
axes[1, 1].set_xlabel('Cost (minimize)')
axes[1, 1].set_ylabel('Performance (maximize)')
axes[1, 1].set_title('Objective Space')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nVisualization complete!")
print(f"Total points evaluated: {len(initial_X)}")
print(f"Feasible points: {feasible.sum()}/{len(feasible)}")

## Step 6: Comparison - With vs Without DeterministicModel

In [ ]:
# Demonstrate the difference between using DeterministicModel vs GP for analytical function

# Option 1: Using DeterministicModel (correct approach)
deterministic_model = GenericDeterministicModel(f=analytical_cost_function)

# Option 2: Using GP for analytical function (inefficient approach)
analytical_values = analytical_cost_function(initial_X)
gp_for_analytical = SingleTaskGP(initial_X, analytical_values)

# Test on new points
test_points = torch.tensor([[0.2, 0.7], [0.8, 0.3], [0.5, 0.5]])
true_values = analytical_cost_function(test_points)

# DeterministicModel predictions (exact)
det_predictions = deterministic_model(test_points)

# GP predictions (approximate)
gp_predictions = gp_for_analytical.posterior(test_points)

print("Comparison: DeterministicModel vs GP for Analytical Function")
print("=" * 60)
print(f"{'Point':<15} {'True Value':<12} {'Deterministic':<12} {'GP Mean':<12} {'GP Std':<12}")
print("-" * 60)

for i in range(len(test_points)):
    point_str = f"({test_points[i, 0]:.1f}, {test_points[i, 1]:.1f})"
    true_val = true_values[i].item()
    det_val = det_predictions[i].item()
    gp_mean = gp_predictions.mean[i].item()
    gp_std = gp_predictions.variance[i].sqrt().item()
    
    print(f"{point_str:<15} {true_val:<12.6f} {det_val:<12.6f} {gp_mean:<12.6f} {gp_std:<12.6f}")

# Calculate errors
det_error = torch.abs(det_predictions - true_values).mean()
gp_error = torch.abs(gp_predictions.mean - true_values).mean()

print(f"\nMean Absolute Error:")
print(f"  DeterministicModel: {det_error:.10f}")
print(f"  Gaussian Process:   {gp_error:.6f}")

print(f"\nKey Advantages of DeterministicModel:")
print(f"  1. Exact predictions (no approximation error)")
print(f"  2. No uncertainty where none should exist")
print(f"  3. No need to 'learn' a function you already know")
print(f"  4. Computationally efficient")
print(f"  5. Works with any number of test points (no training data needed)")

## Practical Implementation Tips

In [ ]:
# Tip 1: Handling different input/output dimensions
def multi_output_analytical_function(x: torch.Tensor) -> torch.Tensor:
    """
    Example of analytical function with multiple outputs.
    Returns [cost, efficiency] where both are analytical.
    """
    cost = (x**2).sum(dim=-1, keepdim=True)
    efficiency = 1.0 / (1.0 + cost)  # Higher cost = lower efficiency
    return torch.cat([cost, efficiency], dim=-1)

multi_output_model = GenericDeterministicModel(
    f=multi_output_analytical_function,
    num_outputs=2
)

test_result = multi_output_model(torch.tensor([[0.5, 0.5]]))
print("Multi-output DeterministicModel:")
print(f"  Input: [0.5, 0.5]")
print(f"  Output shape: {test_result.mean.shape}")
print(f"  Cost: {test_result.mean[0, 0].item():.4f}")
print(f"  Efficiency: {test_result.mean[0, 1].item():.4f}")

# Tip 2: Error handling for edge cases
def robust_analytical_function(x: torch.Tensor) -> torch.Tensor:
    """
    Example with input validation and edge case handling.
    """
    if x.dim() != 2 or x.shape[-1] != 2:
        raise ValueError(f"Expected input shape (..., 2), got {x.shape}")
    
    # Clamp inputs to valid range if needed
    x_clamped = torch.clamp(x, min=0.0, max=1.0)
    
    # Your analytical computation
    result = (x_clamped**2).sum(dim=-1, keepdim=True)
    
    return result

robust_model = GenericDeterministicModel(f=robust_analytical_function)
print("\nRobust analytical function handles edge cases.")

# Tip 3: Integration patterns with different acquisition functions
print("\nIntegration Patterns:")
print("1. Single-objective: Use DeterministicModel directly with EI")
print("2. Multi-objective: Combine in ModelList for Pareto optimization")
print("3. Constrained: Use analytical constraints in acquisition functions")
print("4. Mixed models: Combine deterministic + GP models as shown above")

# Tip 4: When NOT to use DeterministicModel
print("\nWhen NOT to use DeterministicModel:")
print("- Function has inherent noise/uncertainty")
print("- Function is not truly analytical (requires approximation)")
print("- You want to model epistemic uncertainty in your 'known' function")
print("- Function evaluation is still expensive (defeats the purpose)")

## Summary

This tutorial demonstrated the key concepts for using `DeterministicModel` in optimization:

### What We Covered

1. **Basic Usage**: Creating `GenericDeterministicModel` from analytical functions
2. **Mixed Models**: Combining deterministic and probabilistic models with `ModelList`
3. **Multi-objective Optimization**: Using mixed models for Pareto optimization
4. **Practical Comparisons**: Showing advantages over using GPs for analytical functions
5. **Implementation Tips**: Handling edge cases and different scenarios

### Key Benefits

- **Efficiency**: No surrogate modeling for known functions
- **Accuracy**: Exact evaluation of analytical functions
- **Flexibility**: Easy integration with existing BoTorch/Ax workflows
- **Scalability**: Works with arbitrary input dimensions and multiple outputs

### Use Cases

This approach is ideal when you have:
- **Analytical cost functions** (monetary, computational, energy)
- **Known physical constraints** (conservation laws, geometric constraints)
- **Mixed optimization problems** with both cheap analytical and expensive black-box functions
- **Multi-objective scenarios** where some objectives are analytically computable

### Integration with Ax

While this tutorial focused on BoTorch components, these patterns can be integrated into Ax through:
- Custom `Surrogate` classes (as sketched in the advanced example)
- Modular BoTorch interface configurations
- Custom `ModelBridge` implementations for more complex scenarios

This minimal working example provides a solid foundation for incorporating analytical knowledge into your Bayesian optimization workflows, addressing the use cases mentioned in GitHub issues #935 and #1192.

### Next Steps

- Extend to your specific analytical functions
- Experiment with different acquisition functions
- Integrate with Ax's high-level APIs
- Consider multi-fidelity scenarios with mixed model types